Integrated Mean Squared Error (IMSE) Example
============================================

This example demonstrates how to use the Integrated Mean Squared Error (IMSE)
acquisition function provided in smt-optim. IMSE is often used in
Stepwise Uncertainty Reduction (SUR) to globally minimize surrogate uncertainty.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from smt.surrogate_models import KRG
from smt_optim.acquisition_functions.integrated_mean_squared_error import integrated_mean_squared_error

np.random.seed(42)

# Define a 1D test function
def f(x):
    return (x * 6 - 2)**2 * np.sin(x * 12 - 4)

# Training data (3 points)
xt = np.array([[0.0], [0.5], [1.0]])
yt = f(xt)

# Train the Kriging model
sm = KRG(theta0=[1e-2], print_global=False)
sm.set_training_values(xt, yt)
sm.train()

# Define Monte-Carlo integration points over the domain [0, 1]
integration_points = np.linspace(0, 1, 100).reshape(-1, 1)

# Evaluate IMSE for candidate points across the domain
candidate_points = np.linspace(0, 1, 50).reshape(-1, 1)
imse_values = integrated_mean_squared_error(
    sm, 
    candidate_points, 
    integration_points, 
    inv_block=True
)

# Plotting the result
fig, ax1 = plt.subplots(figsize=(8, 5))

# Plot the surrogate mean and variance
x_plot = np.linspace(0, 1, 200).reshape(-1, 1)
y_mean = sm.predict_values(x_plot)
y_var = sm.predict_variances(x_plot)
y_std = np.sqrt(np.maximum(y_var, 0))

ax1.plot(x_plot, y_mean, 'b-', label='Surrogate Mean')
ax1.fill_between(x_plot.flatten(), 
                 (y_mean - 3 * y_std).flatten(), 
                 (y_mean + 3 * y_std).flatten(), 
                 color='b', alpha=0.2, label='99% CI')
ax1.plot(xt, yt, 'ro', label='Training Points')
ax1.set_xlabel('x')
ax1.set_ylabel('Objective Function', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.legend(loc='upper left')

# Plot IMSE on a secondary axis
ax2 = ax1.twinx()
ax2.plot(candidate_points, imse_values, 'g--', label='IMSE')
ax2.set_ylabel('IMSE Value', color='g')
ax2.tick_params(axis='y', labelcolor='g')
ax2.legend(loc='upper right')

plt.title('Integrated Mean Squared Error (IMSE) Acquisition')
plt.show()